# B5: Forest Harvest (monthly)

Estimate monthly forest-logging HANPPharv.
- Source: FAO forestry statistics + forest cover
- Method: allocate annual fellings (uniform or seasonally adjusted)
- Output: monthly forest HANPPharv (g C m⁻²) → `interim/`

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray
from geocube.api.core import make_geocube
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))
from grid_utils import create_target_grid

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

In [ ]:
gdf_world_merged = (
    gpd.read_parquet(cfg["paths"]["output"] / "world_clean.parquet")
        .query("NE_name != 'Antarctica'")
        .reset_index(drop=True)
        .assign(idx=lambda x: np.arange(1, x.shape[0]+1))    
)
da_template = create_target_grid()
da_countries = make_geocube(gdf_world_merged, measurements=["idx"], like=da_template)["idx"].fillna(0)

# FAO data

In [ ]:
_years = list(range(2010, 2025))
FAO_m49_lst = gdf_world_merged['FAO_m49'].dropna().unique().tolist()
year_cols = [f"Y{y}" for y in _years]
meta_cols = ["Area Code", "Area Code (M49)", "Area", "Item Code", "code", "Item", "Element Code", "Element", "Unit"]
dic_year_rename = {f"Y{y}": y for y in _years}
dic_FAO2region = gdf_world_merged.set_index("FAO_name")["PNAS_region"].to_dict()

In [ ]:
wood_lst = [
    "Industrial roundwood, coniferous",      # 1627
    "Industrial roundwood, non-coniferous",  # 1628
    "Wood fuel, coniferous",                 # 1866
    "Wood fuel, non-coniferous",             # 1867
]

df_wood_all = (
    pd.read_csv(cfg["paths"]["fao"] / "2024/Forestry/Forestry_E_All_Data_NOFLAG.csv", encoding_errors="ignore")
    .query("Element == 'Production'")
    .query("Item in @wood_lst")
    .query("`Area Code (M49)` in @FAO_m49_lst")
    [["Area", "Item"] + year_cols]
    .rename(columns=dic_year_rename)
    .reset_index(drop=True)
    .assign(PNAS_region=lambda x: x["Area"].map(dic_FAO2region))
    .assign(
        stream=lambda x: x["Item"].str.contains("Industrial").map({True: "industrial", False: "woodfuel"}),
        Item=lambda x: x["Item"].str.extract(r"(coniferous|non-coniferous)")[0],
    )
)

## FAO wood relationships

## Split industrial / woodfuel

In [ ]:
df_forest_recover_rate = pd.DataFrame([
    ["N. Africa W. Asia", 0.75, 0.75], ["Sub-Saharan Africa", 0.75, 0.81], ["C. Asia", 0.75, 0.69], ["E. Asia", 0.75, 0.69], ["S. Asia", 0.75, 0.69], ["N. America", 0.92, 0.89], ["Latin America", 0.75, 0.84], ["W. Europe", 0.85, 0.86], ["E. Europe", 0.81, 0.79], ["Oceania", 0.92, 0.89]
], columns=["PNAS_region", "coniferous", "non-coniferous"]).melt(id_vars=["PNAS_region"])\
    .rename(columns={"variable": "Item", "value": "rec_r"})

df_forest_wood_rate = pd.DataFrame([
    ["N. Africa W. Asia", 0.42, 0.57], ["Sub-Saharan Africa", 0.43, 0.57], ["C. Asia", 0.41, 0.57], ["E. Asia", 0.41, 0.57], ["S. Asia", 0.41, 0.57], ["N. America", 0.43, 0.60], ["Latin America", 0.43, 0.60], ["W. Europe", 0.41, 0.57], ["E. Europe", 0.41, 0.57], ["Oceania", 0.41, 0.57]
], columns=["PNAS_region", "coniferous", "non-coniferous"]).melt(id_vars=["PNAS_region"])\
    .rename(columns={"variable": "Item", "value": "wood_r"})
    

In [ ]:
# -- Industrial roundwood: needs rec_r / wood_r conversion --
    
df_industrial_C = (
    df_wood_all.query("stream == 'industrial'")
    .merge(df_forest_recover_rate, on=["PNAS_region", "Item"], how="left")
    .merge(df_forest_wood_rate,    on=["PNAS_region", "Item"], how="left")
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        df[_years].fillna(0)
            .divide(df["rec_r"], axis=0)
            .mul(df["wood_r"], axis=0)
            .mul(1e6).mul(0.5)
    ], axis=1))
    .drop(columns=["rec_r", "wood_r", "stream"])
    .set_index("Area").join(gdf_world_merged.set_index("FAO_name")["idx"]).reset_index()
)

# -- Wood fuel: direct m3 x density x C; rec_r/wood_r not applicable --
# wood_r still needed (stem fraction); rec_r not needed (woodfuel is itself Removals ~ Fellings)
df_woodfuel_C = (
    df_wood_all.query("stream == 'woodfuel'")
    .merge(df_forest_wood_rate, on=["PNAS_region", "Item"], how="left")
    .pipe(lambda df: pd.concat([
        df.drop(columns=_years),
        df[_years].fillna(0)
            .mul(df["wood_r"], axis=0)
            .mul(1e6).mul(0.5)
    ], axis=1))
    .drop(columns=["wood_r", "stream"])
    .set_index("Area").join(gdf_world_merged.set_index("FAO_name")["idx"]).reset_index()
)

In [ ]:
year = 2015

fao_ind_conifer = (
    df_industrial_C
    .query("Item == 'coniferous'")
    .set_index("idx")[year]
    .to_dict()
)
fao_ind_broadleaf = (
    df_industrial_C
    .query("Item == 'non-coniferous'")
    .set_index("idx")[year]
    .to_dict()
)

fao_wf_conifer = (
    df_woodfuel_C
    .query("Item == 'coniferous'")
    .set_index("idx")[year]
    .to_dict()
)
fao_wf_broadleaf = (
    df_woodfuel_C
    .query("Item == 'non-coniferous'")
    .set_index("idx")[year]
    .to_dict()
)


## Without industrial/woodfuel split (deprecated)

# Logging mask

# Tree-type fractions

## Exclude shrubs

In [ ]:
da_logging_mask = xr.open_dataarray(cfg["paths"]["gee_exports"] / f"logging_mask-1000m.tif")\
    .sel(band=1).drop_vars("band")\
    .drop_attrs()\
    .fillna(0).astype(int)\
    .rio.write_crs("EPSG:4326")


In [ ]:
lc = (
    xr.open_dataarray(cfg["paths"]["gee_exports"] / f"luc/luc-500m.tif")
    .chunk({"band": 1, "x": 2200, "y": 2200}
    )
    .assign_coords(band=range(2001, 2025))
    .rename({"band": "year"})
    .rio.write_crs("EPSG:4326")
    .sel(year=slice(2010, 2024))
    .astype(np.uint8)
)

In [ ]:
da_logging_mask_reproj = da_logging_mask\
    .rio.reproject_match(lc.isel(year=0), resampling=Resampling.nearest)

In [ ]:
da_logging_mask_reproj = da_logging_mask_reproj.chunk({"x": 2200, "y": 2200})

In [ ]:
NEEDLE_TYPES    = [1, 3]   # Evergreen/Deciduous Needleleaf
BROADLEAF_TYPES = [2, 4]   # Evergreen/Deciduous Broadleaf
MIXED_TYPE      = 5        # Mixed Forest -> 0.5 each

In [ ]:
da_WDPA_lst = []
for year in range(2010, 2025):
    da_WDPA = xr.open_dataarray(cfg["paths"]["output"] / f"protected_areas/WDPA_{year}.tif", chunks={"x": 2200, "y": 2200})\
        .sel(band=1).drop_vars("band").fillna(0).astype(np.uint8)\
        .drop_attrs()
    da_WDPA_lst.append(da_WDPA)
da_WDPA = xr.concat(da_WDPA_lst, dim="year").assign_coords(year=range(2010, 2025))

In [ ]:
chunks = {"year": 1, "y": 22 * 100, "x": 22 * 100}   # ~5 MB uint8 per chunk
lc = lc.chunk(chunks)
da_WDPA = da_WDPA.chunk(chunks)
da_logging_mask_reproj = da_logging_mask_reproj.chunk(
    {"y": 22 * 100, "x": 22 * 100}
)

### Coniferous

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
needle_score = (
    lc.isin(NEEDLE_TYPES).astype(np.uint8) * np.uint8(2)
    + (lc == MIXED_TYPE).astype(np.uint8)
)  # still uint8, max value 2


In [ ]:
pieces = []
for t in tqdm(list(range(lc.sizes["year"]))):
    piece = (
        (needle_score.isel(year=t)
         * da_logging_mask_reproj
         * da_WDPA.isel(year=t))
        .coarsen(x=22, y=22, boundary="trim")
        .mean()
    ).compute()
    pieces.append(piece)

needle_frac = (xr.concat(pieces, dim="year") / 2.0).astype(np.float32)

In [ ]:
needle_frac_proj = needle_frac.rio.reproject_match(da_template, resampling=Resampling.nearest)

In [ ]:
needle_frac_proj.to_dataset(name="needle_frac")\
    .to_netcdf(
        cfg["paths"]["output"] / f"WDPA_masked/needle_frac_logging_masked.nc", 
        encoding={"needle_frac": {"dtype": "float32", "zlib": True, "complevel": 4}})

### Broadleaf

In [ ]:
broadleaf_score = (
    lc.isin(BROADLEAF_TYPES).astype(np.uint8) * np.uint8(2)
    + (lc == MIXED_TYPE).astype(np.uint8)
)  # still uint8, max value 2

In [ ]:
pieces = []
for t in tqdm(list(range(lc.sizes["year"]))):
    piece = (
        (broadleaf_score.isel(year=t)
         * da_logging_mask_reproj
         * da_WDPA.isel(year=t))
        .coarsen(x=22, y=22, boundary="trim")
        .mean()
    ).compute()
    pieces.append(piece)

broadleaf_frac = (xr.concat(pieces, dim="year") / 2.0).astype(np.float32)

In [ ]:
broadleaf_frac_proj = broadleaf_frac.rio.reproject_match(da_template, resampling=Resampling.nearest)

In [ ]:
broadleaf_frac_proj.to_dataset(name="broadleaf_frac")\
    .to_netcdf(
        cfg["paths"]["output"] / f"WDPA_masked/broadleaf_frac_logging_masked.nc", 
        encoding={"broadleaf_frac": {"dtype": "float32", "zlib": True, "complevel": 4}})

## Include shrubs

In [ ]:
NEEDLE_TYPES    = [1, 3]   # Evergreen/Deciduous Needleleaf
BROADLEAF_TYPES = [2, 4, 6, 7, 8, 9]   # Evergreen/Deciduous Broadleaf
MIXED_TYPE      = 5        # Mixed Forest -> 0.5 each

### Coniferous forest

In [ ]:
needle_score = (
    lc.isin(NEEDLE_TYPES).astype(np.uint8) * np.uint8(2)
    + (lc == MIXED_TYPE).astype(np.uint8)
)  # still uint8, max value 2

In [ ]:
pieces = []
for t in tqdm(list(range(lc.sizes["year"]))):
    piece = (
        (needle_score.isel(year=t)
         * da_WDPA.isel(year=t))
        .coarsen(x=22, y=22, boundary="trim")
        .mean()
    ).compute()
    pieces.append(piece)

needle_frac = (xr.concat(pieces, dim="year") / 2.0).astype(np.float32)

In [ ]:
needle_frac_proj = needle_frac.rio.reproject_match(da_template, resampling=Resampling.nearest)

In [ ]:
needle_frac_proj.to_dataset(name="needle_frac")\
    .to_netcdf(
        cfg["paths"]["output"] / f"WDPA_masked/needle_frac_including_shrub.nc", 
        encoding={"needle_frac": {"dtype": "float32", "zlib": True, "complevel": 4}})

### Broadleaf forest

In [ ]:
broadleaf_score = (
    lc.isin(BROADLEAF_TYPES).astype(np.uint8) * np.uint8(2)
    + (lc == MIXED_TYPE).astype(np.uint8)
)  # still uint8, max value 2

In [ ]:
pieces = []
for t in tqdm(list(range(lc.sizes["year"]))):
    piece = (
        (broadleaf_score.isel(year=t)
         * da_WDPA.isel(year=t))
        .coarsen(x=22, y=22, boundary="trim")
        .mean()
    ).compute()
    pieces.append(piece)

broadleaf_frac = (xr.concat(pieces, dim="year") / 2.0).astype(np.float32)

In [ ]:
broadleaf_frac_proj = broadleaf_frac.rio.reproject_match(da_template, resampling=Resampling.nearest)

In [ ]:
broadleaf_frac_proj.to_dataset(name="broadleaf_frac")\
    .to_netcdf(
        cfg["paths"]["output"] / f"WDPA_masked/broadleaf_frac_including_shrub.nc", 
        encoding={"broadleaf_frac": {"dtype": "float32", "zlib": True, "complevel": 4}})

## Total vegetation cover fraction

In [ ]:
VEGETATED_ALL = [1,2,3,4,5,6,7,8,9,10,11,12,14]

In [ ]:
veg_score = (
    lc.isin(VEGETATED_ALL).astype(np.uint8)
)

In [ ]:
pieces = []
for t in tqdm(list(range(lc.sizes["year"]))):
    piece = (
        (veg_score.isel(year=t)
         * da_WDPA.isel(year=t))
        .coarsen(x=22, y=22, boundary="trim")
        .mean()
    ).compute()
    pieces.append(piece)

veg_frac = (xr.concat(pieces, dim="year") / 2.0).astype(np.float32)

In [ ]:
veg_frac_proj = veg_frac.rio.reproject_match(da_template, resampling=Resampling.nearest)

In [ ]:
veg_frac_proj.to_dataset(name="veg_frac")\
    .to_netcdf(
        cfg["paths"]["output"] / f"WDPA_masked/veg_frac.nc", 
        encoding={"veg_frac": {"dtype": "float32", "zlib": True, "complevel": 4}})

## IQR-based revised woodfuel weights

For each forest type, countries above the upper 1.5-IQR fence of annual woodfuel carbon per mapped forest area are reweighted with total vegetation cover, matching the executed workflow described in the manuscript.


In [ ]:
fraction_root = cfg["paths"]["output"] / "WDPA_masked"
needle_with_shrub = xr.open_dataset(
    fraction_root / "needle_frac_including_shrub.nc"
)["needle_frac"]
broadleaf_with_shrub = xr.open_dataset(
    fraction_root / "broadleaf_frac_including_shrub.nc"
)["broadleaf_frac"]
vegetation_fraction = xr.open_dataset(
    fraction_root / "veg_frac.nc"
)["veg_frac"]

country_values = da_countries.fillna(0).values.astype(np.intp)
flat_country = country_values.ravel()
number_of_ids = int(country_values.max()) + 1

country_needle_area = np.bincount(
    flat_country,
    weights=(
        needle_with_shrub.mean(dim="year").values * da_template.values
    ).ravel(),
    minlength=number_of_ids,
)
country_broadleaf_area = np.bincount(
    flat_country,
    weights=(
        broadleaf_with_shrub.mean(dim="year").values * da_template.values
    ).ravel(),
    minlength=number_of_ids,
)

woodfuel_needle = (
    df_woodfuel_C.query("Item == 'coniferous'")
    .drop(columns=["Item", "PNAS_region", "Area"])
    .set_index("idx")
)
woodfuel_broadleaf = (
    df_woodfuel_C.query("Item == 'non-coniferous'")
    .drop(columns=["Item", "PNAS_region", "Area"])
    .set_index("idx")
)

needle_intensity = np.full((number_of_ids, len(_years)), np.nan)
broadleaf_intensity = np.full((number_of_ids, len(_years)), np.nan)
for column_index, year_value in enumerate(_years):
    needle_total = np.zeros(number_of_ids)
    for code, value in woodfuel_needle[year_value].to_dict().items():
        if 0 <= int(code) < number_of_ids:
            needle_total[int(code)] = value
    valid = (needle_total > 0) & (country_needle_area > 0)
    needle_intensity[valid, column_index] = (
        needle_total[valid] / country_needle_area[valid]
    )

    broadleaf_total = np.zeros(number_of_ids)
    for code, value in woodfuel_broadleaf[year_value].to_dict().items():
        if 0 <= int(code) < number_of_ids:
            broadleaf_total[int(code)] = value
    valid = (broadleaf_total > 0) & (country_broadleaf_area > 0)
    broadleaf_intensity[valid, column_index] = (
        broadleaf_total[valid] / country_broadleaf_area[valid]
    )


def get_outlier_ids(intensity, number_of_ids):
    """Return countries above the upper 1.5-IQR fence in any year."""
    values = intensity[np.isfinite(intensity)]
    q1, q3 = np.nanpercentile(values, [25, 75])
    upper_fence = q3 + 1.5 * (q3 - q1)
    outliers = set(
        np.where(np.any(intensity > upper_fence, axis=1))[0]
    )
    return outliers, upper_fence


needle_outliers, needle_fence = get_outlier_ids(
    needle_intensity, number_of_ids
)
broadleaf_outliers, broadleaf_fence = get_outlier_ids(
    broadleaf_intensity, number_of_ids
)
print(
    "Woodfuel IQR revisions:",
    f"coniferous={len(needle_outliers)} countries,",
    f"broadleaf={len(broadleaf_outliers)} countries",
)


In [ ]:
needle_outlier_mask = np.isin(
    country_values, list(needle_outliers)
)[np.newaxis, :, :]
broadleaf_outlier_mask = np.isin(
    country_values, list(broadleaf_outliers)
)[np.newaxis, :, :]

needle_frac_revised = xr.DataArray(
    np.where(
        needle_outlier_mask,
        vegetation_fraction,
        needle_with_shrub,
    ),
    coords=vegetation_fraction.coords,
    dims=vegetation_fraction.dims,
)
broadleaf_frac_revised = xr.DataArray(
    np.where(
        broadleaf_outlier_mask,
        vegetation_fraction,
        broadleaf_with_shrub,
    ),
    coords=vegetation_fraction.coords,
    dims=vegetation_fraction.dims,
)

needle_frac_revised.to_dataset(name="needle_frac").to_netcdf(
    fraction_root / "needle_frac_revised.nc",
    encoding={
        "needle_frac": {
            "dtype": "float32", "zlib": True, "complevel": 4
        }
    },
)
broadleaf_frac_revised.to_dataset(name="broadleaf_frac").to_netcdf(
    fraction_root / "broadleaf_frac_revised.nc",
    encoding={
        "broadleaf_frac": {
            "dtype": "float32", "zlib": True, "complevel": 4
        }
    },
)


# Allocator

In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path
from rasterio.enums import Resampling


class ForestAllocator:
    """
    Forest HANPPharv allocator: uniform monthly harvest + coniferous/broadleaf two-stream allocation.

    temporal weight: 1/12 per month (uniform-logging assumption)
    spatial weight: logging_mask x coniferous/broadleaf area ratio x annual NPP

    FAO two streams:
        Roundwood coniferous     (Item 1862) -> coniferous weight layer
        Roundwood non-coniferous (Item 1863) -> broadleaf weight layer

    data sources:
        PSN          : GEE-exported MOD17A2HGF monthly PSN (gC/m2/day)
        logging_mask : Sims et al. 2025 classification==4, 1km → 0.1°
        needle_frac  : MODIS MCD12Q1 LC_Type1, classes 1+3 (+0.5x5)
        broadleaf_frac: MODIS MCD12Q1, classes 2+4 (+0.5x5)
        NPP_annual   : monthly-summed PSN proxy (consistent with the cropland pipeline)
    """


    def __init__(self, cfg: dict, year: int, mode: str = "industrial"):
        """
        mode : "industrial" -> logging_masked weight
            "woodfuel"   -> revised weight for IQR-flagged countries
        """
        self.year = year
        self.mode = mode

        self.npp_path = (Path(cfg["paths"]["gee_exports"])
                        / f"NPP_annual_2010_2025.tif")

        suffix = "logging_masked" if mode == "industrial" else "revised"
        self.lc_needle_frac    = Path(cfg["paths"]["output"]) / f"WDPA_masked/needle_frac_{suffix}.nc"
        self.lc_broadleaf_frac = Path(cfg["paths"]["output"]) / f"WDPA_masked/broadleaf_frac_{suffix}.nc"

        self.da_template = create_target_grid()
        self.target_y    = self.da_template.y.values
        self.target_x    = self.da_template.x.values
        self.grid_area   = self.da_template.values

        self.npp_annual     = None
        self.needle_frac    = None
        self.broadleaf_frac = None
    # ==================================================================
    # load data
    # ==================================================================
    
    def load(self):
        self._load_npp()
        self._load_lc_fractions()

    def _load_npp(self):
        self.npp_annual = (
            xr.open_dataarray(self.npp_path)
            .chunk({"x": 1000, "y": 1000})
            .assign_coords(band=range(2010, 2026))
            .rename({"band": "year"})
            .sel(year=self.year)
            .drop_attrs()
            .rio.write_crs("EPSG:4326")
            .clip(min=0)
            .rio.reproject_match(self.da_template, resampling=Resampling.nearest)
            .fillna(0)
            .values   # (ny, nx)
        )

    def _load_lc_fractions(self):
        self.needle_frac = (
            xr.open_dataset(self.lc_needle_frac)
            ["needle_frac"].sel(year=self.year).drop_vars("year").drop_attrs()
        )
        self.broadleaf_frac = (
            xr.open_dataset(self.lc_broadleaf_frac)
            ["broadleaf_frac"].sel(year=self.year).drop_vars("year").drop_attrs()
        )
        
    # ==================================================================
    # build weights
    # ==================================================================

    def build_harvest_weight(self):
        npp_annual = self.npp_annual   # (ny, nx)

        base_n = (self.needle_frac.values    * self.grid_area * npp_annual)
        base_b = (self.broadleaf_frac.values * self.grid_area * npp_annual)

        hw_n = np.broadcast_to(base_n[np.newaxis] / 12.0,
                                (12,) + base_n.shape).copy().astype(np.float64)
        hw_b = np.broadcast_to(base_b[np.newaxis] / 12.0,
                                (12,) + base_b.shape).copy().astype(np.float64)
        return hw_n, hw_b
    
    # ==================================================================
    # allocate
    # ==================================================================

    def allocate(self, fao_conifer: dict,
                 fao_broadleaf: dict,
                 country_mask: xr.DataArray) -> xr.DataArray:
        """
        Parameters
        ----------
        fao_conifer   : {country_idx: annual carbon gC}  <- FAO Item 1862
        fao_broadleaf : {country_idx: annual carbon gC}  <- FAO Item 1863
        country_mask  : DataArray (ny, nx)

        Returns
        -------
        DataArray (12, ny, nx)  forest HANPPharv (gC)
        """
        hw_n, hw_b = self.build_harvest_weight()
        mask       = country_mask.fillna(0).values.astype(np.intp)
        ny, nx     = mask.shape
        n_ids      = int(mask.max()) + 1
        flat_mask  = mask.ravel()

        def _allocate_one(hw: np.ndarray, fao: dict) -> np.ndarray:
            country_total = np.zeros(n_ids, dtype=np.float64)
            for mo in range(12):
                country_total += np.bincount(
                    flat_mask, weights=hw[mo].ravel(), minlength=n_ids
                )
            fao_arr = np.zeros(n_ids, dtype=np.float64)
            for code, val in fao.items():
                if 0 <= int(code) < n_ids:
                    fao_arr[int(code)] = val

            safe_total = np.where(country_total > 0, country_total, 1.0)
            ratio      = (fao_arr / safe_total)[flat_mask].reshape(ny, nx)
            return hw * ratio[np.newaxis, :, :]

        result = (_allocate_one(hw_n, fao_conifer)
                  + _allocate_one(hw_b, fao_broadleaf))

        return xr.DataArray(
            result.astype(np.float32),
            dims=["month", "y", "x"],
            coords={"month": np.arange(1, 13),
                    "y": self.target_y,
                    "x": self.target_x},
            name="forest_HANPPharv",
        )

In [ ]:
year = 2015

fao_ind_conifer = (
    df_industrial_C
    .query("Item == 'coniferous'")
    .set_index("idx")[year]
    .to_dict()
)
fao_ind_broadleaf = (
    df_industrial_C
    .query("Item == 'non-coniferous'")
    .set_index("idx")[year]
    .to_dict()
)

fao_wf_conifer = (
    df_woodfuel_C
    .query("Item == 'coniferous'")
    .set_index("idx")[year]
    .to_dict()
)
fao_wf_broadleaf = (
    df_woodfuel_C
    .query("Item == 'non-coniferous'")
    .set_index("idx")[year]
    .to_dict()
)


fa_ind = ForestAllocator(cfg, year=year, mode="industrial")
fa_ind.load()
result_industrial = fa_ind.allocate(fao_ind_conifer, fao_ind_broadleaf, da_countries)

fa_wf = ForestAllocator(cfg, year=year, mode="woodfuel")
fa_wf.load()
result_woodfuel = fa_wf.allocate(fao_wf_conifer, fao_wf_broadleaf, da_countries)

da_forest_harvest = (result_industrial + result_woodfuel) / da_template

In [ ]:
def cal_harv_1year(year):
    fao_ind_conifer = (
        df_industrial_C
        .query("Item == 'coniferous'")
        .set_index("idx")[year]
        .to_dict()
    )
    fao_ind_broadleaf = (
        df_industrial_C
        .query("Item == 'non-coniferous'")
        .set_index("idx")[year]
        .to_dict()
    )

    fao_wf_conifer = (
        df_woodfuel_C
        .query("Item == 'coniferous'")
        .set_index("idx")[year]
        .to_dict()
    )
    fao_wf_broadleaf = (
        df_woodfuel_C
        .query("Item == 'non-coniferous'")
        .set_index("idx")[year]
        .to_dict()
    )

    fa_ind = ForestAllocator(cfg, year=year, mode="industrial")
    fa_ind.load()
    result_industrial = fa_ind.allocate(fao_ind_conifer, fao_ind_broadleaf, da_countries)

    fa_wf = ForestAllocator(cfg, year=year, mode="woodfuel")
    fa_wf.load()
    result_woodfuel = fa_wf.allocate(fao_wf_conifer, fao_wf_broadleaf, da_countries)

    da_HANPPharv = (result_industrial + result_woodfuel) / da_template
    ds_HANPPharv = xr.Dataset({"forest": da_HANPPharv})
    
    # save
    encoding = {
        var: {"dtype": "float32", "zlib": True, "complevel": 4, "chunksizes": (1, 180, 360)}
        for var in ds_HANPPharv.data_vars
    }
    ds_HANPPharv.to_netcdf(
        Path(cfg["paths"]["output"]) / f"HANPPharv_forest_{year}.nc",
        encoding=encoding
    )

In [ ]:
for year in range(2010, 2025):
    if (cfg["paths"]["output"] / f"HANPPharv_forest_{year}.nc").exists():
        continue
    cal_harv_1year(year)